In [23]:
mdata["methyl"].obs_names

Index(['subject2', 'subject3', 'subject4', 'subject5', 'subject6', 'subject9',
       'subject10', 'subject11', 'subject12', 'subject15', 'subject17',
       'subject21', 'subject23', 'subject24', 'subject26', 'subject27',
       'subject29', 'subject30', 'subject31', 'subject33', 'subject34',
       'subject36', 'subject37', 'subject39', 'subject40', 'subject43',
       'subject45', 'subject46', 'subject47', 'subject50'],
      dtype='object', name='sample_names')

In [2]:
mdata = mudata.read("sim-data_n-50_effect-2_sigma-indep_corr-0.5.h5mu")

/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/mudata/_core/mudata.py:491: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(


In [15]:
d = set(["a", "sample_name"])

In [16]:
d

{'a', 'sample_name'}

In [18]:
if 'sample_name' in d:
    print("Tr")

Tr


In [19]:
a

,sample_name,response,sample_names
sample_names,,,
subject2,subject2,no,subject2
subject3,subject3,no,subject3
subject4,subject4,no,subject4
subject5,subject5,no,subject5
subject6,subject6,no,subject6
subject9,subject9,no,subject9
subject10,subject10,no,subject10
subject11,subject11,no,subject11
subject12,subject12,no,subject12


In [12]:
if 'sample_names' in obs_cols:
    bb = aa.rename(columns={'sample_names': 'sample_name'}, errors='ignore')
    
    # Ensure only sample_name column is present if both exist
if 'sample_names' in obs_cols and 'sample_name' in obs_cols:
    bb = aa.drop(columns=['sample_names'])
    

In [13]:
bb

,sample_name,response
sample_names,,
subject2,subject2,no
subject3,subject3,no
subject4,subject4,no
subject5,subject5,no
subject6,subject6,no
subject9,subject9,no
subject10,subject10,no
subject11,subject11,no
subject12,subject12,no


In [8]:
aa = mdata["methyl"].obs

In [10]:
obs_cols = aa.columns

In [11]:
obs_cols

Index(['sample_name', 'response', 'sample_names'], dtype='object')

In [4]:
mdata.obs

,methyl:sample_name,methyl:response,methyl:sample_names,expr:sample_name,expr:response,expr:sample_names,protein:sample_name,protein:response,protein:sample_names
sample_names,,,,,,,,,
subject2,subject2,no,subject2,subject2,no,subject2,subject2,no,subject2
subject3,subject3,no,subject3,subject3,no,subject3,subject3,no,subject3
subject4,subject4,no,subject4,subject4,no,subject4,subject4,no,subject4
subject5,subject5,no,subject5,subject5,no,subject5,subject5,no,subject5
subject6,subject6,no,subject6,subject6,no,subject6,subject6,no,subject6
subject9,subject9,no,subject9,subject9,no,subject9,subject9,no,subject9
subject10,subject10,no,subject10,subject10,no,subject10,subject10,no,subject10
subject11,subject11,no,subject11,subject11,no,subject11,subject11,no,subject11
subject12,subject12,no,subject12,subject12,no,subject12,subject12,no,subject12


In [1]:
# Imports goes here
from docopt import docopt
import os
import numpy as np
import pandas as pd
import mudata
# Custom script import
from load_test_splits   import  load_test_splits
from tr_te_mdata        import  tr_te_mdata
from save_metadata      import  save_metadata
from save_blocks_labels import  save_blocks_labels
from merge_obs_metadata import  merge_obs_metadata
# For each split of the folds do the following

# This is the main runner
def prepare_mogonet_input(mdata_path, splits_dir, base_dir):
    mdata = mudata.read(mdata_path)
    # Notice index here is 0-based (read in txts)
    test_splits_df = load_test_splits(splits_dir=splits_dir)
    obs_df = merge_obs_metadata(mdata.obs)
    for i, split in enumerate(test_splits_df.index):
        # Create output folder first
        outdir = f"{base_dir}_{i+1}"
        os.makedirs(outdir, exist_ok=True)
        # Note the obs metadata is going to have repeated info, given
        # each block has the same sample_names, response, age, so need to combine those
        # obs_df = merge_obs_metadata(mdata.obs)
        # Then need to split blocks and   labels
        train, test, meta_test = tr_te_mdata(mdata, test_splits_df, meta=obs_df, split=split)
        # For each train and test we need to split the blocks out and their labels
        # The input is always comes in X , y combination
        print(f"Saving metadata for test set for {split}")
        save_metadata(meta_test, i+1, outdir)
        print(f"Saving blocks and labels for {split}")
        save_blocks_labels(train, test, outdir)
    print(f"Done preparing mogonet inputs")
    return "Done"

# # Execute it here
# if __name__ == '__main__':
#   # Parse docopt
#   args = docopt(__doc__)
#   # Execute runner
#   prepare_mogonet_input(
#     mdata_path      = args["--mdata_path"],
#     splits_dir      = args["--splits_dir"],
#     base_dir        = args["--base_dir"]
#   )